In [1]:
import os, sys
sys.path.append(os.path.abspath('..'))

In [ ]:
# Library imports
import json
import time
import numpy as np
import pandas as pd
import torch
import gymnasium as gym

from env import InventoryEnv
from stable_baselines3 import DQN
from stable_baselines3.dqn.policies import DQNPolicy, QNetwork

# Large but finite - NOT -inf. The Bellman target in DQN.train() computes
# next_q_values.max(dim=1) and multiplies it by (1 - dones); for a terminal
# transition every action is masked (episode only ends when every warehouse hits 0
# capacity), so max(dim=1) would be -inf and (1 - dones) is 0 for that transition -
# 0 * -inf is NaN, not 0, which corrupts the whole network within a few updates.
# A large finite sentinel makes 0 * sentinel = 0 as expected, while still being far
# below any real Q-value in this environment's reward scale (worst case, for the
# largest instances trained here, is on the order of -1e5 - see
# num_customers_options/num_warehouses_options below - so -1e8 keeps a comfortable
# margin without risking float32 overflow if it ever ends up in a squared-error term).
MASK_VALUE = -1e8

# Same normalization constant env.py's get_state() already uses for warehouses_distance
# (max warehouse-to-customer distance for the placements in env.py, e.g. (-50,-50) to a
# (100,100) corner is sqrt(150**2 + 150**2)). Reused here to rescale the reward fed to
# the learner - NOT applied inside env.py itself, since env.step()'s raw distance reward
# is also the evaluation metric reported throughout export_results.ipynb (real total
# distance, matching the paper's Eq. 9 units); rescaling it there would silently change
# the units of every results table. A uniform positive rescale of every step's reward
# doesn't change the optimal policy (trajectory ordering, and hence the argmax policy,
# is preserved under multiplication by a positive constant) - it just keeps returns
# closer to the O(1) scale PPO's critic and DQN's Q-regression assume under
# stable-baselines3's default hyperparameters (learning rate, network init), instead of
# the O(1e5) raw cumulative distance for the larger instances trained here.
DISTANCE_NORM = 212.13


class ScaledRewardWrapper(gym.RewardWrapper):
    """Divides every step's reward by DISTANCE_NORM before the learning algorithm sees
    it. See DISTANCE_NORM's comment for why this is done here instead of in env.py."""

    def reward(self, reward):
        return reward / DISTANCE_NORM


def _capacity_mask(obs, num_warehouses):
    """Per env.py's default get_state layout, obs[..., 2*i] is warehouse i's normalized
    distance and obs[..., 2*i+1] is its normalized remaining capacity - which is exactly
    0 iff that warehouse is depleted. Works for both a single observation and a batch
    (any number of leading dims), and for both numpy arrays and torch tensors."""
    capacity = obs[..., 1::2][..., :num_warehouses]
    return capacity > 0


class MaskedQNetwork(QNetwork):
    """Q-network that masks out depleted warehouses (0 remaining capacity) before
    action selection. Used for both q_net and q_net_target (see MaskedDQNPolicy), so
    masking applies to greedy action selection AND to the max_a' Q(s', a') term in the
    Bellman target computed in DQN.train()."""

    def forward(self, obs):
        q_values = super().forward(obs)
        mask = _capacity_mask(obs, q_values.shape[-1])
        return q_values.masked_fill(~mask, MASK_VALUE)


class MaskedDQNPolicy(DQNPolicy):
    def make_q_net(self):
        net_args = self._update_features_extractor(self.net_args, features_extractor=None)
        return MaskedQNetwork(**net_args).to(self.device)


class MaskedDQN(DQN):
    """Vanilla stable-baselines3 DQN, with the one addition needed now that env.step()
    raises instead of silently rerouting on a 0-capacity warehouse: action selection
    must never propose one. Two call sites do this without going through the (masked)
    Q-network at all - epsilon-greedy random exploration and the initial warmup phase -
    so both are overridden here to sample only among warehouses with capacity > 0.
    Greedy/exploitation action selection needs no override: it already goes through
    self.policy.predict() -> q_net(obs), which MaskedQNetwork masks."""

    def _sample_action(self, learning_starts, action_noise=None, n_envs=1):
        assert self._last_obs is not None, "self._last_obs was not set"
        if self.num_timesteps < learning_starts and not (self.use_sde and self.use_sde_at_warmup):
            mask = _capacity_mask(np.asarray(self._last_obs), self.action_space.n)
            unscaled_action = np.array([np.random.choice(np.flatnonzero(row)) for row in mask])
        else:
            unscaled_action, _ = self.predict(self._last_obs, deterministic=False)

        # discrete action space: no scaling/clipping needed (mirrors the discrete branch
        # of OffPolicyAlgorithm._sample_action)
        buffer_action = unscaled_action
        action = buffer_action
        return action, buffer_action

    def predict(self, observation, state=None, episode_start=None, deterministic=False):
        if not deterministic and np.random.rand() < self.exploration_rate:
            mask = _capacity_mask(np.asarray(observation), self.action_space.n)
            if mask.ndim == 1:
                action = np.array(np.random.choice(np.flatnonzero(mask)))
            else:
                action = np.array([np.random.choice(np.flatnonzero(row)) for row in mask])
        else:
            action, state = self.policy.predict(observation, state, episode_start, deterministic)
        return action, state

In [3]:
%load_ext tensorboard
%tensorboard --logdir='deep_q_networks_training/Data_Training' --port 6001    #Change port if needed (6006,9009,9999)
#pip install --upgrade tensorboard tensorflow

In [ ]:
def train_dqn(num_warehouses, num_customers, capacity_distribution):

    # Deliberate reseed. np.random.seed also covers env.create_customer() (numpy's
    # *global* RNG, since seed= passed to MaskedDQN() below only seeds SB3's own
    # internals) and MaskedDQN._sample_action/predict's own np.random.choice() calls
    # above, which bypass SB3's internal (seed=-scoped) RNG entirely.
    SEED = 42
    np.random.seed(SEED)

    env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)
    env = ScaledRewardWrapper(env)

    env.reset()

    model_DQN = MaskedDQN(MaskedDQNPolicy,
                    env, 
                    tensorboard_log=f"deep_q_networks_training/Data_Training/DQN/w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}/", 
                    verbose=0,
                    seed=SEED,  # seeds SB3's own internals (policy init, action_space sampling, replay buffer)
                    #exploration_fraction=0.8,
                    #exploration_initial_eps=1,
                    #exploration_final_eps=0.01,
                )

    n_steps = num_customers * 50_000    #10_000

    start_time = time.perf_counter()
    model_DQN.learn(n_steps, reset_num_timesteps=True)
    training_time_minutes = (time.perf_counter() - start_time) / 60

    model_DQN.save(f'deep_q_networks_training/rl_models/dqn_models/dqn_model_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.zip')
    model_DQN.save_replay_buffer(f'deep_q_networks_training/rl_models/dqn_replay_buffer/replay_buffer_dqn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.zip')
    print('Saved DQN model')

    return training_time_minutes

# NOTE: even with the above seeded, exact reproducibility across machines/time also
# needs pinned package versions (torch, stable-baselines3, CUDA/cuDNN - requirements.txt
# already pins the Python package versions) and the same GPU architecture/driver, since
# cuDNN's deterministic algorithms are not guaranteed bit-identical across different
# GPU models - not addressed here.

In [ ]:
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
capacity_distribution_options = ['uniform', 'uneven']

training_times = []  # one row per (num_warehouses, num_customers, capacity_distribution) family

for num_customers in num_customers_options:
    for num_warehouses in num_warehouses_options:
        for capacity_distribution in capacity_distribution_options:
            training_time_minutes = train_dqn(num_warehouses, num_customers, capacity_distribution)
            print(f"Trained model for {num_warehouses} warehouses, {num_customers} customers, {capacity_distribution} capacity")

            training_times.append({
                'num_warehouses': num_warehouses,
                'num_customers': num_customers,
                'capacity_distribution': capacity_distribution,
                'training_time': round(training_time_minutes, 2),
            })

info_dir = 'deep_q_networks_training'
os.makedirs(info_dir, exist_ok=True)

training_times_df = pd.DataFrame(training_times)
training_times_df.to_csv(os.path.join(info_dir, 'deep_q_networks_training_times.csv'), index=False, float_format='%.5f')